In [1]:
import qi
import os
import sys
import time
import threading

from naoqi import ALProxy

# Bot's IP address and port
ip = '10.104.23.217'
port = 9559

In [2]:
file_manager = ALProxy("ALFileManager", ip, port)
shared_folder_name = file_manager.getUserSharedFolderPath()
shared_folder_name

'/home/nao/'

In [3]:
!pip install paramiko
import paramiko
def transfer_file(remote_path="/home/nao/microphones/recording.wav", local_path="pepper-bot/recordings/recording.wav"):
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(ip, username='nao', password='nao')

    sftp = ssh.open_sftp()
    sftp.get(remote_path, local_path)

    sftp.close()
    ssh.close()

def receive_file(local_path="pepper-bot/recordings/recording.wav", remote_path="/home/nao/microphones/recording.wav"):
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(ip, username='nao', password='nao')

    sftp = ssh.open_sftp()
    sftp.put(local_path, remote_path)

    sftp.close()
    ssh.close()

DEPRECATION: Python 2.7 reached the end of its life on January 1st, 2020. Please upgrade your Python as Python 2.7 is no longer maintained. A future version of pip will drop support for Python 2.7. More details about Python 2 support in pip, can be found at https://pip.pypa.io/en/latest/development/release-process/#python-2-support
     |████████████████████████████████| 213 kB 6.2 MB/s eta 0:00:01
     |████████████████████████████████| 2.6 MB 6.3 MB/s eta 0:00:01
     |████████████████████████████████| 964 kB 6.3 MB/s eta 0:00:01
     |████████████████████████████████| 59 kB 630 kB/s eta 0:00:011
     |████████████████████████████████| 390 kB 5.8 MB/s eta 0:00:01
     |████████████████████████████████| 118 kB 8.8 MB/s eta 0:00:01
You should consider upgrading via the '/usr/local/bin/python -m pip install --upgrade pip' command.


/usr/local/lib/python2.7/site-packages/paramiko/transport.py:33: CryptographyDeprecationWarning: Python 2 is no longer supported by the Python core team. Support for it is now deprecated in cryptography, and will be removed in the next release.
  from cryptography.hazmat.backends import default_backend


In [19]:
def eye_color(status="ListenOn", rgb = None):
    leds = ALProxy("ALLeds", ip, port)

    if rgb is None:
        if status == "ListenOn":
            leds.fadeRGB("AllLeds", "Yellow", 1)
        elif status == "SpeechDetected":
            leds.fadeRGB("AllLeds", "Green", 8)
        elif status == "EndOfProcess":
            leds.fadeRGB("AllLeds", "White", 1)

    else:
        leds.fadeRGB("AllLeds", rgb, 1)


# Callback function for timer
def timer_callback(timer_cb):
    timer_cb = True

# Records Audio when speech is detected
def record_audio(timer=8, path_name="/home/nao/microphones/recording.wav", debug=False):
    path_name = os.path.join(path_name)
    # Create Connection Proxies
    recorder = ALProxy("ALAudioRecorder", ip, port)
    speech_recogniser = ALProxy("ALSpeechRecognition", ip, port)
    sound_detector = ALProxy("ALSoundDetection", ip, port)
    memory = ALProxy("ALMemory", ip, port)
    
    timer_cb = False

    speech_recogniser.subscribe("speech_recognition")
    speech_recogniser.setLanguage("English")
    vocabulary = ["pepper", "hello"]
    speech_recogniser.setVocabulary(vocabulary, False)

    # Stop recording when speech stops being detected or timer is reached
    if timer is not None:
        threading.Timer(timer, timer_callback, timer_cb).start()

    # Start recording
    while True:
        time.sleep(0.5)
        status = memory.getData("ALSpeechRecognition/Status")

        # Print status
        if debug:
            print(status)

        # Start recording when speech is detected
        if status == "SpeechDetected":
            eye_color(status)
            print("Recording is starting...")
            recorder.startMicrophonesRecording(path_name, "wav", 16000, (0,0,1,0))
            break

        # Stop recording when timer is reached
        if timer_cb:
            break

    if timer_cb:
        return

    while True:
        time.sleep(0.5)
        status = memory.getData("ALSpeechRecognition/Status")

        # Stop recording when speech stops being detected
        if status == "EndOfProcess":
            print("Recording is stopping...")
            recorder.stopMicrophonesRecording()
            speech_recogniser.unsubscribe("speech_recognition")
            eye_color(status)
            break

        # Stop recording when timer is reached
        if timer_cb:
            recorder.stopMicrophonesRecording()
            speech_recogniser.unsubscribe("speech_recognition")
            leds = ALProxy("ALLeds", ip, port)
            leds.fadeRGB("AllLeds", "White", 1)
            break


def record_audio_sd(timer=None, path_name="/home/nao/microphones/recording.wav", debug=False):
    path_name = os.path.join(path_name)
    # Create Connection Proxies
    recorder = ALProxy("ALAudioRecorder", ip, port)
    sound_detector = ALProxy("ALSoundDetection", ip, port)
    memory = ALProxy("ALMemory", ip, port)
    
    timer_cb = False

    sound_detector.subscribe("sound_detector")
    sound_detector.setParameter("Sensitivity", 0.9)

    # Record audio when sound is detected
    if timer is not None:
        threading.Timer(timer, timer_callback, timer_cb).start()

    # Start recording
    while True:
        time.sleep(0.5)
        status = memory.getData("SoundDetected")

        # Print status
        if debug:
            print(status)

        # Start recording when sound is detected
        if status is not None:
            print("Recording is starting...")
            recorder.startMicrophonesRecording(path_name, "wav", 16000, (0,0,1,0))
            break

        # Stop recording when timer is reached
        if timer_cb:
            break

    if timer_cb:
        return
    
    while True:
        sound_detector.setParameter("Sensitivity", 0.8)

        time.sleep(2)
        status = memory.getData("SoundDetected")

        # Stop recording when sound stops being detected
        if status is None:
            recorder.stopMicrophonesRecording()
            sound_detector.unsubscribe("sound_detector")
            leds = ALProxy("ALLeds", ip, port)
            leds.fadeRGB("AllLeds", "White", 1)
            break

        # Stop recording when timer is reached
        if timer_cb:
            recorder.stopMicrophonesRecording()
            sound_detector.unsubscribe("sound_detector")
            leds = ALProxy("ALLeds", ip, port)
            leds.fadeRGB("AllLeds", "White", 1)
            break

In [16]:
status = memory.getData("SoundDetected")
print(status)

[4523, 0, 1.0, 0]


In [ ]:
recorder = ALProxy("ALAudioRecorder", ip, port)
recorder.stopMicrophonesRecording()

In [18]:
record_audio_sd(timer=10, debug=True)
transfer_file()

# Sam, put your code here, also your functions here too

[[7623, 0, 1.0, 0]]
Recording is starting...


Exception in thread Thread-4:
Traceback (most recent call last):
  File "/usr/local/lib/python2.7/threading.py", line 801, in __bootstrap_inner
    self.run()
  File "/usr/local/lib/python2.7/threading.py", line 1072, in run
    self.function(*self.args, **self.kwargs)
TypeError: timer_callback() takes exactly 1 argument (0 given)



KeyboardInterrupt: 

In [ ]:
# Record Video
def record_video(timer=10):
    video_recorder = ALProxy("ALVideoRecorder", ip, port)
    video_recorder.startRecording("/home/nao/recordings/cameras", "test_video")
    time.sleep(timer)
    videoInfo = video_recorder.stopRecording()

    print('Video was saved on the robot: ', videoInfo[1])
    print('Total number of frames: ', videoInfo[0])

('Video was saved on the robot: ', '/home/nao/recordings/cameras/test_video.avi')
('Total number of frames: ', 73)


**Method 2**

In [ ]:
!pip install pyftplib
import pyftplib
from pyftplib import FTPServer
from pyftplib.handlers import FTPHandler
from pyftplib.authorizers import DummyAuthorizer
import threading

def ftp_transfer_file():
    remote_path = "/home/nao/microphones/recording.wav"
    authorizer = DummyAuthorizer()
    authorizer.add_user("nao", "nao", remote_path, perm="elradfmw")
    handler = FTPHandler
    handler.authorizer = authorizer
    server = FTPServer((ip, 21), handler)
    server_thread = threading.Thread(target=server.serve_forever)
    server_thread.daemon = True
    server_thread.start()

# Tablet

In [23]:
tablet_service = ALProxy("ALTabletService", ip, port)
tablet_service.wakeUp()

# Put code for tablet web view here